In [ ]:
import matplotlib.pyplot as plt
import matplotlib.widgets as widgets
import numpy as np
from matplotlib.collections import QuadMesh
from matplotlib.image import AxesImage

def interactive_colorbar_sliders(fig, ax_array):
    """
    Add interactive sliders to control colorbar limits for all color plots in the given axes.
    
    Parameters:
    -----------
    fig : matplotlib.figure.Figure
        The matplotlib figure containing the plots
    ax_array : numpy.ndarray or matplotlib.axes.Axes or list
        Single axis, array of axes, or list of axes containing color plots
        
    Returns:
    --------
    dict : Dictionary containing slider widgets for programmatic control if needed
    """
    
    # Ensure ax_array is iterable
    if hasattr(ax_array, 'flat'):
        # Handle numpy array of axes
        axes = ax_array.flat
    elif hasattr(ax_array, '__iter__') and not hasattr(ax_array, 'plot'):
        # Handle list of axes
        axes = ax_array
    else:
        # Handle single axis
        axes = [ax_array]
    
    # Find all color mappable objects (images, contours, etc.)
    color_objects = []
    data_ranges = []
    
    for ax in axes:
        for child in ax.get_children():
            if isinstance(child, (QuadMesh, AxesImage)) or hasattr(child, 'get_array'):
                try:
                    data = child.get_array()
                    if data is not None and len(data) > 0:
                        # Remove masked values for range calculation
                        if hasattr(data, 'compressed'):
                            valid_data = data.compressed()
                        else:
                            valid_data = data[~np.isnan(data)] if np.any(np.isnan(data)) else data
                        
                        if len(valid_data) > 0:
                            color_objects.append(child)
                            data_ranges.append((np.min(valid_data), np.max(valid_data)))
                except:
                    continue
    
    if not color_objects:
        print("No color mappable objects found in the provided axes.")
        return {}
    
    # Calculate global data range
    all_mins = [r[0] for r in data_ranges]
    all_maxs = [r[1] for r in data_ranges]
    global_min = np.min(all_mins)
    global_max = np.max(all_maxs)
    
    # Get current colorbar limits (use first object as reference)
    current_vmin, current_vmax = color_objects[0].get_clim()
    
    # If current limits are the default, use data range
    if current_vmin is None:
        current_vmin = global_min
    if current_vmax is None:
        current_vmax = global_max
    
    # Adjust figure to make room for sliders
    plt.subplots_adjust(bottom=0.25)
    
    # Create slider axes
    slider_height = 0.03
    slider_spacing = 0.05
    
    ax_vmin = plt.axes([0.2, 0.1, 0.5, slider_height])
    ax_vmax = plt.axes([0.2, 0.1 - slider_spacing, 0.5, slider_height])
    
    # Create sliders
    slider_vmin = widgets.Slider(
        ax_vmin, 'Min', 
        global_min, global_max, 
        valinit=current_vmin,
        valfmt='%.3f'
    )
    
    slider_vmax = widgets.Slider(
        ax_vmax, 'Max', 
        global_min, global_max, 
        valinit=current_vmax,
        valfmt='%.3f'
    )
    
    # Update function
    def update_colorbar(val=None):
        vmin = slider_vmin.val
        vmax = slider_vmax.val
        
        # Ensure vmin < vmax
        if vmin >= vmax:
            if val == vmin:  # vmin slider was moved
                vmax = vmin + 0.001 * (global_max - global_min)
                slider_vmax.set_val(vmax)
            else:  # vmax slider was moved
                vmin = vmax - 0.001 * (global_max - global_min)
                slider_vmin.set_val(vmin)
        
        # Update all color mappable objects
        for obj in color_objects:
            obj.set_clim(vmin, vmax)
        
        # Redraw the figure
        fig.canvas.draw_idle()
    
    # Connect sliders to update function
    slider_vmin.on_changed(lambda val: update_colorbar(val))
    slider_vmax.on_changed(lambda val: update_colorbar(val))
    
    # Add reset button
    ax_reset = plt.axes([0.8, 0.1 - slider_spacing/2, 0.1, slider_height])
    button_reset = widgets.Button(ax_reset, 'Reset')
    
    def reset_sliders(event):
        slider_vmin.reset()
        slider_vmax.reset()
        update_colorbar()
    
    button_reset.on_clicked(reset_sliders)
    
    # Return slider objects for potential programmatic control
    return {
        'vmin_slider': slider_vmin,
        'vmax_slider': slider_vmax,
        'reset_button': button_reset,
        'update_function': update_colorbar
    }


# Example usage function
def demo_interactive_colorbar():
    """
    Demonstration of the interactive colorbar slider function
    """
    # Create sample data
    x = np.linspace(-3, 3, 100)
    y = np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(x, y)
    Z1 = np.exp(-(X**2 + Y**2))
    Z2 = np.sin(X) * np.cos(Y)
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Create color plots
    im1 = axes[0].imshow(Z1, extent=[-3, 3, -3, 3], origin='lower', cmap='viridis')
    axes[0].set_title('Gaussian')
    plt.colorbar(im1, ax=axes[0])
    
    im2 = axes[1].imshow(Z2, extent=[-3, 3, -3, 3], origin='lower', cmap='plasma')
    axes[1].set_title('Sin x Cos')
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    
    # Add interactive sliders
    sliders = interactive_colorbar_sliders(fig, axes)
    
    return fig, axes, sliders

# Uncomment the following lines to run the demo:
%matplotlib widget
fig, axes, sliders = demo_interactive_colorbar()
plt.show()

In [ ]:
# Enable interactive backend
%matplotlib widget
# %matplotlibjupyter

# Create your figure and plots
fig, ax = plt.subplots()
im = ax.imshow(your_data, cmap='viridis')
plt.colorbar(im)

# Add interactive sliders
sliders = interactive_colorbar_sliders(fig, ax)
plt.show()

In [ ]:
!jupyter labextension install @jupyter-widgets/jupyterlab-manager

In [ ]:
# Check what backends are available
import matplotlib
print(matplotlib.get_backend())
print("Available backends:", matplotlib.rcsetup.interactive_bk)

In [ ]:
%matplotlib notebook

# Now create your plot
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots()
data = np.random.randn(50, 50)  # Replace with your data
im = ax.imshow(data, cmap='viridis')
plt.colorbar(im)

# Use the original matplotlib widgets version
sliders = interactive_colorbar_sliders(fig, ax)
plt.show()

In [ ]:
%matplotlib nbagg

# Create your plot
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots()
data = np.random.randn(50, 50)  # Replace with your data
im = ax.imshow(data, cmap='viridis')
plt.colorbar(im)
plt.show()

# Add interactive controls
controls = interactive_colorbar_ipywidgets(fig, ax)

In [ ]:
%matplotlib nbagg

In [ ]:
%matplotlib inline
# ... your plotting code ...
controls = interactive_colorbar_ipywidgets(fig, ax)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.widgets as widgets
import numpy as np
from matplotlib.collections import QuadMesh
from matplotlib.image import AxesImage

def interactive_colorbar_ipywidgets(fig, ax_array):
    """
    Alternative version using ipywidgets instead of matplotlib widgets.
    Works with %matplotlib inline backend.
    """
    try:
        from ipywidgets import interact, FloatSlider, Button, VBox, HBox
        from IPython.display import display
    except ImportError:
        print("ipywidgets not available. Please install with: pip install ipywidgets")
        return {}
    
    # Ensure ax_array is iterable
    if hasattr(ax_array, 'flat'):
        axes = ax_array.flat
    elif hasattr(ax_array, '__iter__') and not hasattr(ax_array, 'plot'):
        axes = ax_array
    else:
        axes = [ax_array]
    
    # Find all color mappable objects
    color_objects = []
    data_ranges = []
    
    for ax in axes:
        for child in ax.get_children():
            if isinstance(child, (QuadMesh, AxesImage)) or hasattr(child, 'get_array'):
                try:
                    data = child.get_array()
                    if data is not None and len(data) > 0:
                        if hasattr(data, 'compressed'):
                            valid_data = data.compressed()
                        else:
                            valid_data = data[~np.isnan(data)] if np.any(np.isnan(data)) else data
                        
                        if len(valid_data) > 0:
                            color_objects.append(child)
                            data_ranges.append((np.min(valid_data), np.max(valid_data)))
                except:
                    continue
    
    if not color_objects:
        print("No color mappable objects found.")
        return {}
    
    # Calculate ranges
    all_mins = [r[0] for r in data_ranges]
    all_maxs = [r[1] for r in data_ranges]
    global_min = np.min(all_mins)
    global_max = np.max(all_maxs)
    
    current_vmin, current_vmax = color_objects[0].get_clim()
    if current_vmin is None:
        current_vmin = global_min
    if current_vmax is None:
        current_vmax = global_max
    
    # Create sliders
    vmin_slider = FloatSlider(
        value=current_vmin,
        min=global_min,
        max=global_max,
        step=(global_max - global_min) / 100,
        description='Min:'
    )
    
    vmax_slider = FloatSlider(
        value=current_vmax,
        min=global_min,
        max=global_max,
        step=(global_max - global_min) / 100,
        description='Max:'
    )
    
    def update_colorbar(vmin, vmax):
        # Ensure vmin < vmax
        if vmin >= vmax:
            vmax = vmin + 0.001 * (global_max - global_min)
        
        for obj in color_objects:
            obj.set_clim(vmin, vmax)
        fig.canvas.draw()
    
    # Interactive widget
    widget = interact(update_colorbar, 
                     vmin=vmin_slider, 
                     vmax=vmax_slider)
    
    return {'widget': widget, 'vmin_slider': vmin_slider, 'vmax_slider': vmax_slider}

In [ ]:
%matplotlib inline

# Create your plot
fig, ax = plt.subplots()
data = np.random.randn(50, 50)  # Replace with your actual data
im = ax.imshow(data, cmap='viridis')
plt.colorbar(im)
plt.show()

# Add interactive controls
controls = interactive_colorbar_ipywidgets(fig, ax)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.widgets as widgets
import numpy as np
from matplotlib.collections import QuadMesh
from matplotlib.image import AxesImage

def interactive_colorbar_sliders(fig, ax_array):
    """
    Add interactive sliders to control colorbar limits for all color plots in the given axes.
    
    Parameters:
    -----------
    fig : matplotlib.figure.Figure
        The matplotlib figure containing the plots
    ax_array : numpy.ndarray or matplotlib.axes.Axes or list
        Single axis, array of axes, or list of axes containing color plots
        
    Returns:
    --------
    dict : Dictionary containing slider widgets for programmatic control if needed
    """
    
    # Ensure ax_array is iterable
    if hasattr(ax_array, 'flat'):
        # Handle numpy array of axes
        axes = ax_array.flat
    elif hasattr(ax_array, '__iter__') and not hasattr(ax_array, 'plot'):
        # Handle list of axes
        axes = ax_array
    else:
        # Handle single axis
        axes = [ax_array]
    
    # Find all color mappable objects (images, contours, etc.)
    color_objects = []
    data_ranges = []
    
    for ax in axes:
        for child in ax.get_children():
            if isinstance(child, (QuadMesh, AxesImage)) or hasattr(child, 'get_array'):
                try:
                    data = child.get_array()
                    if data is not None and len(data) > 0:
                        # Remove masked values for range calculation
                        if hasattr(data, 'compressed'):
                            valid_data = data.compressed()
                        else:
                            valid_data = data[~np.isnan(data)] if np.any(np.isnan(data)) else data
                        
                        if len(valid_data) > 0:
                            color_objects.append(child)
                            data_ranges.append((np.min(valid_data), np.max(valid_data)))
                except:
                    continue
    
    if not color_objects:
        print("No color mappable objects found in the provided axes.")
        return {}
    
    # Calculate global data range
    all_mins = [r[0] for r in data_ranges]
    all_maxs = [r[1] for r in data_ranges]
    global_min = np.min(all_mins)
    global_max = np.max(all_maxs)
    
    # Get current colorbar limits (use first object as reference)
    current_vmin, current_vmax = color_objects[0].get_clim()
    
    # If current limits are the default, use data range
    if current_vmin is None:
        current_vmin = global_min
    if current_vmax is None:
        current_vmax = global_max
    
    # Adjust figure to make room for sliders
    plt.subplots_adjust(bottom=0.25)
    
    # Create slider axes
    slider_height = 0.03
    slider_spacing = 0.05
    
    ax_vmin = plt.axes([0.2, 0.1, 0.5, slider_height])
    ax_vmax = plt.axes([0.2, 0.1 - slider_spacing, 0.5, slider_height])
    
    # Create sliders
    slider_vmin = widgets.Slider(
        ax_vmin, 'Min', 
        global_min, global_max, 
        valinit=current_vmin,
        valfmt='%.3f'
    )
    
    slider_vmax = widgets.Slider(
        ax_vmax, 'Max', 
        global_min, global_max, 
        valinit=current_vmax,
        valfmt='%.3f'
    )
    
    # Update function
    def update_colorbar(val=None):
        vmin = slider_vmin.val
        vmax = slider_vmax.val
        
        # Ensure vmin < vmax
        if vmin >= vmax:
            if val == vmin:  # vmin slider was moved
                vmax = vmin + 0.001 * (global_max - global_min)
                slider_vmax.set_val(vmax)
            else:  # vmax slider was moved
                vmin = vmax - 0.001 * (global_max - global_min)
                slider_vmin.set_val(vmin)
        
        # Update all color mappable objects
        for obj in color_objects:
            obj.set_clim(vmin, vmax)
        
        # Redraw the figure
        fig.canvas.draw_idle()
    
    # Connect sliders to update function
    slider_vmin.on_changed(lambda val: update_colorbar(val))
    slider_vmax.on_changed(lambda val: update_colorbar(val))
    
    # Add reset button
    ax_reset = plt.axes([0.8, 0.1 - slider_spacing/2, 0.1, slider_height])
    button_reset = widgets.Button(ax_reset, 'Reset')
    
    def reset_sliders(event):
        slider_vmin.reset()
        slider_vmax.reset()
        update_colorbar()
    
    button_reset.on_clicked(reset_sliders)
    
    # Return slider objects for potential programmatic control
    return {
        'vmin_slider': slider_vmin,
        'vmax_slider': slider_vmax,
        'reset_button': button_reset,
        'update_function': update_colorbar
    }


# Example usage function
def demo_interactive_colorbar():
    """
    Demonstration of the interactive colorbar slider function
    """
    # Create sample data
    x = np.linspace(-3, 3, 100)
    y = np.linspace(-3, 3, 100)
    X, Y = np.meshgrid(x, y)
    Z1 = np.exp(-(X**2 + Y**2))
    Z2 = np.sin(X) * np.cos(Y)
    
    # Create figure with subplots
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    
    # Create color plots
    im1 = axes[0].imshow(Z1, extent=[-3, 3, -3, 3], origin='lower', cmap='viridis')
    axes[0].set_title('Gaussian')
    plt.colorbar(im1, ax=axes[0])
    
    im2 = axes[1].imshow(Z2, extent=[-3, 3, -3, 3], origin='lower', cmap='plasma')
    axes[1].set_title('Sin x Cos')
    plt.colorbar(im2, ax=axes[1])
    
    plt.tight_layout()
    
    # Add interactive sliders
    sliders = interactive_colorbar_sliders(fig, axes)
    
    return fig, axes, sliders

def interactive_colorbar_ipywidgets(fig, ax_array):
    """
    Alternative version using ipywidgets instead of matplotlib widgets.
    Works with %matplotlib inline backend.
    """
    try:
        from ipywidgets import interact, FloatSlider, Button, VBox, HBox
        from IPython.display import display
    except ImportError:
        print("ipywidgets not available. Please install with: pip install ipywidgets")
        return interactive_colorbar_sliders(fig, ax_array)
    
    # Ensure ax_array is iterable
    if hasattr(ax_array, 'flat'):
        axes = ax_array.flat
    elif hasattr(ax_array, '__iter__') and not hasattr(ax_array, 'plot'):
        axes = ax_array
    else:
        axes = [ax_array]
    
    # Find all color mappable objects
    color_objects = []
    data_ranges = []
    
    for ax in axes:
        for child in ax.get_children():
            if isinstance(child, (QuadMesh, AxesImage)) or hasattr(child, 'get_array'):
                try:
                    data = child.get_array()
                    if data is not None and len(data) > 0:
                        if hasattr(data, 'compressed'):
                            valid_data = data.compressed()
                        else:
                            valid_data = data[~np.isnan(data)] if np.any(np.isnan(data)) else data
                        
                        if len(valid_data) > 0:
                            color_objects.append(child)
                            data_ranges.append((np.min(valid_data), np.max(valid_data)))
                except:
                    continue
    
    if not color_objects:
        print("No color mappable objects found.")
        return {}
    
    # Calculate ranges
    all_mins = [r[0] for r in data_ranges]
    all_maxs = [r[1] for r in data_ranges]
    global_min = np.min(all_mins)
    global_max = np.max(all_maxs)
    
    current_vmin, current_vmax = color_objects[0].get_clim()
    if current_vmin is None:
        current_vmin = global_min
    if current_vmax is None:
        current_vmax = global_max
    
    # Create sliders
    vmin_slider = FloatSlider(
        value=current_vmin,
        min=global_min,
        max=global_max,
        step=(global_max - global_min) / 100,
        description='Min:'
    )
    
    vmax_slider = FloatSlider(
        value=current_vmax,
        min=global_min,
        max=global_max,
        step=(global_max - global_min) / 100,
        description='Max:'
    )
    
    def update_colorbar(vmin, vmax):
        # Ensure vmin < vmax
        if vmin >= vmax:
            vmax = vmin + 0.001 * (global_max - global_min)
        
        for obj in color_objects:
            obj.set_clim(vmin, vmax)
        
        # Try multiple ways to refresh the display
        try:
            fig.canvas.draw()
        except:
            pass
        
        try:
            fig.canvas.draw_idle()
        except:
            pass
            
        # Force display update for inline backend
        try:
            from IPython.display import display, clear_output
            plt.show()
        except:
            pass
    
    # Interactive widget
    widget = interact(update_colorbar, 
                     vmin=vmin_slider, 
                     vmax=vmax_slider)
    
    return {'widget': widget, 'vmin_slider': vmin_slider, 'vmax_slider': vmax_slider}

# Usage examples:
# For ipywidgets version (works with %matplotlib inline):
# %matplotlib inline
# fig, axes, sliders = demo_interactive_colorbar()
# controls = interactive_colorbar_ipywidgets(fig, axes)

# For matplotlib widgets version (requires %matplotlib widget):
# %matplotlib widget  
# fig, axes, sliders = demo_interactive_colorbar()
# plt.show()

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, FloatSlider

# Create your plot
fig, ax = plt.subplots(figsize=(8, 6))
data = np.random.randn(50, 50)  # Replace with your data
im = ax.imshow(data, cmap='viridis')
cbar = plt.colorbar(im)

# Get data range
vmin_orig, vmax_orig = np.min(data), np.max(data)

def update_plot(vmin, vmax):
    # Clear and recreate the plot
    ax.clear()
    im_new = ax.imshow(data, cmap='viridis', vmin=vmin, vmax=vmax)
    ax.set_title(f'Color limits: [{vmin:.2f}, {vmax:.2f}]')
    plt.show()

# Create interactive sliders
interact(update_plot, 
         vmin=FloatSlider(value=vmin_orig, min=vmin_orig, max=vmax_orig, step=(vmax_orig-vmin_orig)/100, description='Min:'),
         vmax=FloatSlider(value=vmax_orig, min=vmin_orig, max=vmax_orig, step=(vmax_orig-vmin_orig)/100, description='Max:'))

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from ipywidgets import interact, FloatSlider
from IPython.display import clear_output

# Your data
data = np.random.randn(50, 50)  # Replace with your actual data
vmin_orig, vmax_orig = np.min(data), np.max(data)

def plot_with_limits(vmin, vmax):
    # Clear previous output
    clear_output(wait=True)
    
    # Create new plot
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(data, cmap='viridis', vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax)
    ax.set_title(f'Colorbar limits: [{vmin:.3f}, {vmax:.3f}]')
    plt.tight_layout()
    plt.show()

# Create interactive widget
interact(plot_with_limits,
         vmin=FloatSlider(
             value=vmin_orig, 
             min=vmin_orig, 
             max=vmax_orig, 
             step=(vmax_orig-vmin_orig)/100, 
             description='Min:'
         ),
         vmax=FloatSlider(
             value=vmax_orig, 
             min=vmin_orig, 
             max=vmax_orig, 
             step=(vmax_orig-vmin_orig)/100, 
             description='Max:'
         ))